# Trading Book Summary
Asset class breakdown, instrument count, and gross/net book size over a period.

In [23]:
import sys
sys.path.insert(0, '../../../')

from datetime import date
import pandas as pd
from importlib import reload
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from reporting.accounting.analysis import month_end_perf as _m 

In [24]:
reload(_m)

<module 'reporting.accounting.analysis.month_end_perf' from '/Users/fabioballoni/Development/repos/risk/risk_pylibrary/reporting/accounting/analysis/../../../reporting/accounting/analysis/month_end_perf.py'>

## Parameters

In [25]:
SDATE      = date(2026, 1, 1)
EDATE      = date(2026, 4, 30)

## Fetch Data

In [26]:
res = _m.trading_book_summary(SDATE, EDATE, fetch_cache=False,verbose=True)


 *** Using cached data (2,139,509 rows)

 *** Asset class breakdown at 2026-04-30

 *** Building daily instrument count

 *** Building book size

   Asset classes : []

   Avg gross book:          9,438,573 EUR

   Avg net book  :          9,161,300 EUR


In [27]:
# Re-run analytics without re-fetching (fetch_cache=False reuses CF.cache_acct_report)
# res = trading_book_summary(SEC_ACC_NO, SDATE, EDATE, fetch_cache=False)

## 1 · Asset Class Breakdown at End Date

In [28]:
breakdown = res['asset_class_breakdown'].reset_index()
breakdown

,sec_acc_no,instrument_type,n_instruments,mkt_eur_net,mkt_eur_gross,mkt_eur_long,mkt_eur_short


In [29]:
fig_breakdown = px.bar(
    breakdown,
    x='instrument_type',
    y=['mkt_eur_long', 'mkt_eur_short'],
    color_discrete_map={'mkt_eur_long': '#2ecc71', 'mkt_eur_short': '#e74c3c'},
    barmode='relative',
    facet_col='sec_acc_no',
    title=f'Asset Class Breakdown — {EDATE}',
    labels={'value': 'Market Value (EUR)', 'instrument_type': 'Asset Class'},
    text_auto='.3s',
    template='plotly_white',
)
fig_breakdown.update_layout(legend_title_text='Direction')
fig_breakdown.show()

## 2 · Daily Instrument Count

In [30]:
instrument_count = res['instrument_count'].reset_index()
instrument_count

,report_date,n_instruments
0,2025-01-01,11871
1,2025-01-02,12469
2,2025-01-03,12470
3,2025-01-06,12474
4,2025-01-07,12487
...,...,...
81,2025-04-24,13227
82,2025-04-25,13229
83,2025-04-28,13237
84,2025-04-29,13239


In [31]:
fig_count = px.line(
    instrument_count,
    x='report_date',
    y='n_instruments',
    title='Daily Instrument Count',
    labels={'report_date': 'Date', 'n_instruments': '# Instruments'},
    markers=True,
    template='plotly_white',
)
fig_count.show()

## 3 · Gross and Net Book Size

In [32]:
daily_book = res['book_size']['daily'].reset_index()
print(f"Avg Gross Book : {res['book_size']['avg_gross']:>18,.0f} EUR")
print(f"Avg Net Book   : {res['book_size']['avg_net']:>18,.0f} EUR")
daily_book

Avg Gross Book :          9,438,573 EUR
Avg Net Book   :          9,161,300 EUR


,report_date,net_book,gross_book
0,2025-01-01,3.751648e+06,4.762264e+06
1,2025-01-02,8.978394e+06,9.945214e+06
2,2025-01-03,8.864468e+06,9.706647e+06
3,2025-01-06,8.817148e+06,9.926598e+06
4,2025-01-07,8.557233e+06,9.774546e+06
...,...,...,...
81,2025-04-24,1.057469e+07,1.077704e+07
82,2025-04-25,1.069682e+07,1.089663e+07
83,2025-04-28,1.061793e+07,1.081746e+07
84,2025-04-29,1.044830e+07,1.065324e+07


In [33]:
fig_book = make_subplots(specs=[[{'secondary_y': True}]])

fig_book.add_trace(
    go.Scatter(x=daily_book['report_date'], y=daily_book['gross_book'],
               name='Gross Book', line=dict(color='#3498db')),
    secondary_y=False,
)
fig_book.add_trace(
    go.Scatter(x=daily_book['report_date'], y=daily_book['net_book'],
               name='Net Book', line=dict(color='#e67e22', dash='dash')),
    secondary_y=False,
)
fig_book.add_hline(
    y=res['book_size']['avg_gross'], line_dash='dot', line_color='#3498db',
    annotation_text='Avg Gross', annotation_position='top left',
)
fig_book.add_hline(
    y=res['book_size']['avg_net'], line_dash='dot', line_color='#e67e22',
    annotation_text='Avg Net', annotation_position='bottom left',
)
fig_book.update_layout(
    title='Daily Gross / Net Book Size',
    xaxis_title='Date',
    yaxis_title='Market Value (EUR)',
    template='plotly_white',
)
fig_book.show()